# Experimento 04 — Estudo Físico Pareado Bulk vs 2D

## Hipótese

Para uma fração relevante dos materiais lamelares ou exfoliáveis presentes em ambas as bases
(Materials Project bulk e C2DB monocamada), o bandgap aumenta na transição 3D → 2D:

> `Δgap = gap_2D − gap_bulk > 0`

Essa variável Δgap é central para entender o viés dos modelos treinados em bulk ao serem
aplicados em materiais 2D.

## Objetivo

Construir um dataset pareado de materiais com versão bulk (MP) e versão 2D (C2DB),
calcular Δgap por composição e gerar análises quantitativas e visuais.

## Critérios de pareamento

- **Pareamento por fórmula reduzida** (`pymatgen.Composition.reduced_formula`)
- Múltiplos polimorfos bulk → preserva todos, anota multiplicidade
- Mesmo critério para múltiplas entradas 2D com mesma fórmula
- Pareamento confiável: apenas materiais com bandgap PBE disponível em ambas as bases
- Pareamento com HSE: subconjunto com gap_hse disponível em ambas as bases

## Referências

- Chen et al. 2019 — MEGNet (`1812.05055v2.pdf`)
- Gjerding et al. 2021 — C2DB
- Ko et al. 2025 — MatGL (`s41524-025-01742-y-1.pdf`)

In [ ]:
from __future__ import annotations

import gzip
import json
import warnings
from pathlib import Path

import ase.db
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pymatgen.core import Structure, Composition
from pymatgen.io.ase import AseAtomsAdaptor

import os
NOTEBOOK_DIR = Path(os.path.abspath(''))

warnings.simplefilter("ignore")
print("Imports OK")
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))

from common import DATA_DIR, FINAL_ROOT, REPO_ROOT, RUNS_DIR, ensure_run_dir, required_path

In [ ]:
# ── Identificação do Run ─────────────────────────────────────────
RUN_ID   = "002"
RUN_NAME = "bulk2d_analysis"
RUN_DIR = ensure_run_dir(RUN_ID, RUN_NAME)

# ── Dados ────────────────────────────────────────────────────────
C2DB_PATH = required_path(DATA_DIR / "raw" / "c2db.db", "C2DB")
MP_STRUCT = required_path(DATA_DIR / "mp.2019.04.01.json", "MP structures")
MP_BGAP   = required_path(DATA_DIR / "band_gap_no_structs.gz", "MP band gaps")
EHULL_MAX = 0.2

print(f"Run   : {RUN_DIR.resolve()}")
print(f"C2DB  : {C2DB_PATH.exists()}")
print(f"MP    : {MP_STRUCT.exists()} / {MP_BGAP.exists()}")

## 1. Carregar C2DB (materiais 2D)

In [ ]:
adaptor = AseAtomsAdaptor()

def load_c2db_for_pairing(db_path: Path, max_ehull: float = 0.2) -> pd.DataFrame:
    """
    Carrega C2DB e retorna DataFrame com:
      uid, formula, reduced_formula, gap_pbe, gap_hse, ehull, dyn_stab
    """
    db = ase.db.connect(str(db_path))
    rows = []
    skipped = 0

    for row in db.select():
        uid   = row.get('uid') or str(row.id)
        ehull = row.get('ehull')
        if ehull is not None and ehull > max_ehull:
            skipped += 1
            continue

        g_pbe = row.get('gap')
        g_hse = row.get('gap_hse')

        if g_pbe is None and g_hse is None:
            skipped += 1
            continue

        try:
            atoms  = row.toatoms()
            struct = adaptor.get_structure(atoms)
            reduced = struct.composition.reduced_formula
            formula = struct.formula
        except Exception:
            skipped += 1
            continue

        rows.append({
            "uid"            : uid,
            "formula"        : formula,
            "reduced_formula": reduced,
            "gap_pbe_2d"     : g_pbe,
            "gap_hse_2d"     : g_hse,
            "ehull"          : ehull,
            "dyn_stab"       : row.get('dyn_stab'),
        })

    df = pd.DataFrame(rows)
    print(f"C2DB carregado    : {len(df)} materiais ({skipped} pulados)")
    print(f"  com PBE         : {df['gap_pbe_2d'].notna().sum()}")
    print(f"  com HSE         : {df['gap_hse_2d'].notna().sum()}")
    print(f"  fórmulas únicas : {df['reduced_formula'].nunique()}")
    return df


df_c2db = load_c2db_for_pairing(C2DB_PATH, max_ehull=EHULL_MAX)

## 2. Carregar Materials Project (bulk 3D)

In [ ]:
def load_mp_for_pairing(struct_path: Path, bgap_path: Path) -> pd.DataFrame:
    """
    Carrega MP.2019.04.01 e retorna DataFrame com:
      material_id, reduced_formula, gap_pbe_bulk, gap_hse_bulk

    Fontes:
    - Estruturas: mp.2019.04.01.json (CIF em string)
    - Bandgaps  : band_gap_no_structs.gz (fidelidades: pbe, hse, gllb-sc, scan)
    """
    # Bandgaps
    print("Carregando bandgaps MP...")
    with gzip.open(str(bgap_path), 'rb') as f:
        bg = json.loads(f.read())
    bg_pbe  = bg.get('pbe', {})
    bg_hse  = bg.get('hse', {})
    useful_ids = set(bg_pbe.keys()) | set(bg_hse.keys())
    print(f"  IDs com PBE  : {len(bg_pbe)}")
    print(f"  IDs com HSE  : {len(bg_hse)}")

    # Estruturas — apenas as que têm bandgap
    print("Carregando estruturas MP (somente com bandgap)...")
    with open(str(struct_path)) as f:
        raw = json.load(f)
    struct_by_id = {
        entry['material_id']: entry['structure']
        for entry in raw
        if entry['material_id'] in useful_ids
    }
    print(f"  Estruturas relevantes: {len(struct_by_id)}")

    rows = []
    skipped = 0
    for mid, cif_str in struct_by_id.items():
        try:
            struct   = Structure.from_str(cif_str, fmt='cif')
            reduced  = struct.composition.reduced_formula
            formula  = struct.formula
        except Exception:
            skipped += 1
            continue

        g_pbe = bg_pbe.get(mid)
        g_hse = bg_hse.get(mid)

        rows.append({
            "material_id"    : mid,
            "formula"        : formula,
            "reduced_formula": reduced,
            "gap_pbe_bulk"   : g_pbe,
            "gap_hse_bulk"   : g_hse,
        })

    df = pd.DataFrame(rows)
    print(f"MP carregado      : {len(df)} entradas ({skipped} puladas)")
    print(f"  com PBE         : {df['gap_pbe_bulk'].notna().sum()}")
    print(f"  com HSE         : {df['gap_hse_bulk'].notna().sum()}")
    print(f"  fórmulas únicas : {df['reduced_formula'].nunique()}")
    return df


df_mp = load_mp_for_pairing(MP_STRUCT, MP_BGAP)

## 3. Pareamento por Fórmula Reduzida

**Critério**: `reduced_formula` igual em C2DB e MP.

Para cada par (2D, bulk) com mesma fórmula:
- Se houver múltiplos polimorfos bulk, conserva todos (explode).
- Se houver múltiplas entradas 2D com a mesma fórmula, conserva todas.
- O campo `n_bulk_matches` indica multiplicidade de polimorfos.

Duas camadas de confiança:
- **PBE pareado**: ambos têm PBE (base larga)
- **HSE pareado**: ambos têm HSE (base restrita)

In [ ]:
def build_paired_dataset(
    df_2d: pd.DataFrame,
    df_bulk: pd.DataFrame,
) -> pd.DataFrame:
    """
    Faz o join por reduced_formula e calcula delta_gap.

    Retorna um DataFrame com uma linha por par (uid_2d, material_id_bulk).
    Para cada par:
      - delta_gap_pbe = gap_pbe_2d - gap_pbe_bulk  (quando ambos disponíveis)
      - delta_gap_hse = gap_hse_2d - gap_hse_bulk  (quando ambos disponíveis)

    Notas sobre viés:
    - Fidelidades diferentes (PBE vs HSE) não são misturadas no cálculo do delta.
    - Para análise cross-fidelidade: coluna delta_gap_pbe2hse disponível separadamente.
    """
    # Agrega bulk por fórmula (pode haver múltiplos polimorfos)
    bulk_by_formula = df_bulk.groupby('reduced_formula').agg(
        gap_pbe_bulk_mean=('gap_pbe_bulk', 'mean'),
        gap_pbe_bulk_min =('gap_pbe_bulk', 'min'),
        gap_pbe_bulk_max =('gap_pbe_bulk', 'max'),
        gap_hse_bulk_mean=('gap_hse_bulk', 'mean'),
        n_bulk_matches   =('material_id',  'count'),
        material_ids     =('material_id',  lambda x: '|'.join(x)),
    ).reset_index()

    # Join
    paired = df_2d.merge(bulk_by_formula, on='reduced_formula', how='inner')

    # Δgap (mesma fidelidade)
    paired['delta_gap_pbe'] = paired['gap_pbe_2d'] - paired['gap_pbe_bulk_mean']
    paired['delta_gap_hse'] = paired['gap_hse_2d'] - paired['gap_hse_bulk_mean']

    # PBE-only e HSE-only válidos
    pbe_valid = paired['gap_pbe_2d'].notna() & paired['gap_pbe_bulk_mean'].notna()
    hse_valid = paired['gap_hse_2d'].notna() & paired['gap_hse_bulk_mean'].notna()

    print(f"Pares totais       : {len(paired)}")
    print(f"  com PBE pareado  : {pbe_valid.sum()}")
    print(f"  com HSE pareado  : {hse_valid.sum()}")
    print(f"  fórmulas únicas  : {paired['reduced_formula'].nunique()}")
    print(f"  multi-bulk (>1)  : {(paired['n_bulk_matches'] > 1).sum()} pares")
    return paired


df_paired = build_paired_dataset(df_c2db, df_mp)
df_paired.head()

In [ ]:
# Subconjuntos para análise
pbe_paired = df_paired.dropna(subset=['gap_pbe_2d', 'gap_pbe_bulk_mean', 'delta_gap_pbe'])
hse_paired = df_paired.dropna(subset=['gap_hse_2d', 'gap_hse_bulk_mean', 'delta_gap_hse'])

print(f"Base PBE pareada : {len(pbe_paired)} pares")
print(f"Base HSE pareada : {len(hse_paired)} pares")

# Estatísticas descritivas de Δgap
print("\n── Δgap PBE (eV) ──")
print(pbe_paired['delta_gap_pbe'].describe().round(3))
frac_pos_pbe = (pbe_paired['delta_gap_pbe'] > 0).mean()
print(f"Fração com Δgap > 0 : {frac_pos_pbe:.1%}")

if len(hse_paired) > 0:
    print("\n── Δgap HSE (eV) ──")
    print(hse_paired['delta_gap_hse'].describe().round(3))
    frac_pos_hse = (hse_paired['delta_gap_hse'] > 0).mean()
    print(f"Fração com Δgap > 0 : {frac_pos_hse:.1%}")

## 4. Análises e Visualizações

In [ ]:
STYLE = dict(linewidth=1.5, edgecolor='white')
FIG_W, FIG_H = 14, 10

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

ax1 = fig.add_subplot(gs[0, 0])  # dist gap bulk PBE
ax2 = fig.add_subplot(gs[0, 1])  # dist gap 2D PBE
ax3 = fig.add_subplot(gs[0, 2])  # scatter bulk vs 2D
ax4 = fig.add_subplot(gs[1, 0])  # hist Δgap PBE
ax5 = fig.add_subplot(gs[1, 1])  # hist Δgap HSE (se disponível)
ax6 = fig.add_subplot(gs[1, 2])  # scatter Δgap PBE vs Δgap HSE

# ── distribuições ────────────────────────────────────────────────
ax1.hist(pbe_paired['gap_pbe_bulk_mean'].dropna(), bins=40, color='steelblue', **STYLE)
ax1.axvline(3.4, color='red', linestyle='--', linewidth=1.2, label='UWBG 3.4 eV')
ax1.set_xlabel('Bandgap bulk PBE (eV)')
ax1.set_ylabel('Contagem')
ax1.set_title('Distribuição – Bulk 3D (PBE)')
ax1.legend(fontsize=8)

ax2.hist(pbe_paired['gap_pbe_2d'].dropna(), bins=40, color='darkorange', **STYLE)
ax2.axvline(3.4, color='red', linestyle='--', linewidth=1.2, label='UWBG 3.4 eV')
ax2.set_xlabel('Bandgap 2D PBE (eV)')
ax2.set_ylabel('Contagem')
ax2.set_title('Distribuição – 2D (PBE)')
ax2.legend(fontsize=8)

ax3.scatter(
    pbe_paired['gap_pbe_bulk_mean'],
    pbe_paired['gap_pbe_2d'],
    alpha=0.3, s=8, color='purple', rasterized=True,
)
lim = max(pbe_paired['gap_pbe_bulk_mean'].max(), pbe_paired['gap_pbe_2d'].max()) + 0.5
ax3.plot([0, lim], [0, lim], 'k--', linewidth=1, label='gap_2D = gap_bulk')
ax3.set_xlabel('Gap bulk PBE (eV)')
ax3.set_ylabel('Gap 2D PBE (eV)')
ax3.set_title('Bulk PBE vs 2D PBE')
ax3.legend(fontsize=8)

ax4.hist(pbe_paired['delta_gap_pbe'], bins=50, color='teal', **STYLE)
ax4.axvline(0, color='black', linestyle='-', linewidth=1.5)
ax4.axvline(pbe_paired['delta_gap_pbe'].mean(), color='red', linestyle='--',
            linewidth=1.2, label=f"média={pbe_paired['delta_gap_pbe'].mean():.2f} eV")
ax4.set_xlabel('Δgap PBE = gap_2D − gap_bulk (eV)')
ax4.set_ylabel('Contagem')
ax4.set_title('Distribuição Δgap (PBE)')
ax4.legend(fontsize=8)

if len(hse_paired) > 10:
    ax5.hist(hse_paired['delta_gap_hse'], bins=30, color='crimson', **STYLE)
    ax5.axvline(0, color='black', linestyle='-', linewidth=1.5)
    ax5.axvline(hse_paired['delta_gap_hse'].mean(), color='navy', linestyle='--',
                linewidth=1.2, label=f"média={hse_paired['delta_gap_hse'].mean():.2f} eV")
    ax5.set_xlabel('Δgap HSE = gap_2D − gap_bulk (eV)')
    ax5.set_ylabel('Contagem')
    ax5.set_title('Distribuição Δgap (HSE)')
    ax5.legend(fontsize=8)
else:
    ax5.text(0.5, 0.5, 'Dados HSE\npareados insuficientes',
             ha='center', va='center', transform=ax5.transAxes, fontsize=12)
    ax5.set_title('Δgap HSE (sem dados)')

# Scatter Δgap PBE vs Δgap HSE
cross = df_paired.dropna(subset=['delta_gap_pbe', 'delta_gap_hse'])
if len(cross) > 5:
    ax6.scatter(cross['delta_gap_pbe'], cross['delta_gap_hse'],
                alpha=0.5, s=12, color='green', rasterized=True)
    lim6 = max(cross[['delta_gap_pbe','delta_gap_hse']].abs().max()) + 0.3
    ax6.plot([-lim6, lim6], [-lim6, lim6], 'k--', linewidth=1)
    ax6.set_xlabel('Δgap PBE (eV)')
    ax6.set_ylabel('Δgap HSE (eV)')
    ax6.set_title('Δgap PBE vs Δgap HSE')
else:
    ax6.text(0.5, 0.5, 'Dados insuficientes\npara correlação',
             ha='center', va='center', transform=ax6.transAxes, fontsize=12)
    ax6.set_title('Δgap PBE vs HSE (sem dados)')

fig.suptitle('Estudo Pareado Bulk 3D → 2D: Variação do Bandgap', fontsize=14, fontweight='bold')
plt.savefig(RUN_DIR / 'figures' / 'paired_analysis_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva.")

In [ ]:
# ── Top materiais com maior Δgap (2D >> bulk) ────────────────────
top_delta = (
    pbe_paired
    .nlargest(20, 'delta_gap_pbe')
    [['uid','reduced_formula','gap_pbe_bulk_mean','gap_pbe_2d','delta_gap_pbe',
      'gap_hse_2d','n_bulk_matches','dyn_stab']]
    .reset_index(drop=True)
)

print("Top 20 materiais com maior Δgap PBE (2D − bulk):")
print(top_delta.to_string(index=False))

In [ ]:
# ── Top materiais com Δgap negativo (bulk > 2D) ──────────────────
neg_delta = (
    pbe_paired
    .nsmallest(10, 'delta_gap_pbe')
    [['uid','reduced_formula','gap_pbe_bulk_mean','gap_pbe_2d','delta_gap_pbe']]
    .reset_index(drop=True)
)
print("Top 10 materiais com Δgap PBE negativo (bulk > 2D):")
print(neg_delta.to_string(index=False))

In [ ]:
# ── Ranking visual dos top 20 Δgap ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

top20 = pbe_paired.nlargest(20, 'delta_gap_pbe')
labels = top20['reduced_formula'] + '\n(' + top20['uid'].str.split('-').str[0] + ')'

bars = ax.barh(range(len(top20)), top20['delta_gap_pbe'], color='teal', edgecolor='white')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Δgap PBE = gap_2D − gap_bulk (eV)', fontsize=11)
ax.set_title('Top 20: Maior Aumento de Bandgap na Transição Bulk → 2D', fontsize=12)
ax.axvline(0, color='black', linewidth=1)

# Anotar valores
for bar, val in zip(bars, top20['delta_gap_pbe']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:+.2f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig(RUN_DIR / 'figures' / 'top20_delta_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Análise na faixa UWBG ────────────────────────────────────────
uwbg_threshold = 3.4

# Materiais que são UWBG em 2D mas não em bulk (transição marcante)
uwbg_only_2d = pbe_paired[
    (pbe_paired['gap_pbe_2d'] >= uwbg_threshold) &
    (pbe_paired['gap_pbe_bulk_mean'] < uwbg_threshold)
]
uwbg_both = pbe_paired[
    (pbe_paired['gap_pbe_2d'] >= uwbg_threshold) &
    (pbe_paired['gap_pbe_bulk_mean'] >= uwbg_threshold)
]
uwbg_only_bulk = pbe_paired[
    (pbe_paired['gap_pbe_2d'] < uwbg_threshold) &
    (pbe_paired['gap_pbe_bulk_mean'] >= uwbg_threshold)
]

print(f"UWBG apenas em 2D (transição bulk→UWBG): {len(uwbg_only_2d)}")
print(f"UWBG em ambos                          : {len(uwbg_both)}")
print(f"UWBG apenas em bulk (downsizing 2D)    : {len(uwbg_only_bulk)}")
print()
print("Materiais que se tornam UWBG apenas no 2D:")
print(uwbg_only_2d[['reduced_formula','gap_pbe_bulk_mean','gap_pbe_2d','delta_gap_pbe']]
      .sort_values('delta_gap_pbe', ascending=False).head(15).to_string(index=False))

## 5. Salvar Outputs

In [ ]:
# Dataset pareado completo
out_paired = RUN_DIR / 'outputs' / 'paired_bulk2d.csv'
df_paired.to_csv(out_paired, index=False)
print(f"Salvo: {out_paired} ({len(df_paired)} linhas)")

# Subconjunto PBE pareado (base para exp 05 e 06)
out_pbe = RUN_DIR / 'outputs' / 'paired_pbe.csv'
pbe_paired.to_csv(out_pbe, index=False)
print(f"Salvo: {out_pbe} ({len(pbe_paired)} linhas)")

# Subconjunto HSE pareado
if len(hse_paired) > 0:
    out_hse = RUN_DIR / 'outputs' / 'paired_hse.csv'
    hse_paired.to_csv(out_hse, index=False)
    print(f"Salvo: {out_hse} ({len(hse_paired)} linhas)")

In [ ]:
# ── Resumo Final ─────────────────────────────────────────────────
print("=" * 55)
print("RESUMO — Experimento 04: Bulk vs 2D Pareado")
print("=" * 55)
print(f"C2DB: {len(df_c2db)} materiais 2D ({df_c2db['reduced_formula'].nunique()} fórmulas)")
print(f"MP  : {len(df_mp)} materiais bulk ({df_mp['reduced_formula'].nunique()} fórmulas)")
print(f"Pares (PBE): {len(pbe_paired)}")
print(f"  Δgap médio (PBE): {pbe_paired['delta_gap_pbe'].mean():.3f} eV")
print(f"  Δgap mediana    : {pbe_paired['delta_gap_pbe'].median():.3f} eV")
print(f"  Fração Δgap > 0 : {(pbe_paired['delta_gap_pbe'] > 0).mean():.1%}")
print()
print("Hipótese física:")
frac_pos = (pbe_paired['delta_gap_pbe'] > 0).mean()
if frac_pos > 0.6:
    print(f"  CONFIRMADA — {frac_pos:.1%} dos materiais têm gap_2D > gap_bulk.")
else:
    print(f"  PARCIALMENTE SUPORTADA — {frac_pos:.1%} dos materiais têm gap_2D > gap_bulk.")
print("=" * 55)

In [ ]:
import shutil

nb_src = NOTEBOOK_DIR / "bulk2d_paired_analysis.ipynb"
nb_dst = RUN_DIR / "notebook.ipynb"
if nb_src.exists():
    shutil.copy2(nb_src, nb_dst)
    print(f"Notebook salvo em: {nb_dst.resolve()}")